In [ ]:
# 2025.12.01 pycaret ensemble Set

In [1]:
import sys

project_root = 'c:/big20/git/big20-ML-project2-team3/CreditCardFraud'

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
import os
import time
import warnings

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# hyperopt 용
from hyperopt               import hp

import HyperParams          as HP 
import utils.data_sampling  as ds 

# 사용자 모듈 사용 가능 (원하면)
from utils import preprocessing as pp
from utils import data_sampling as ds
from utils import user_utils    as uu
from utils import preprocessing as pp
from utils import data_sampling as ds
from utils import model_utils   as mu
from utils import modeling      as mo

from utils.hyperopt_search import hyperopt_search, train_and_evaluate

# PyCaret import
from pycaret.classification import *
from sklearn.svm import SVC

import copy

# HP dict 복사
cat_params = copy.deepcopy(HP.cb_best_params)

# verbose 제거 또는 False로 변경
if 'verbose' in cat_params:
    cat_params['verbose'] = False



In [3]:
# 결과받을 딕셔너리
results = {}
team_rs = 23 # 우리팀 random_state

In [4]:
#1. 데이터 로딩
raw_df = pp.ccf_load_data()

데이터 로드 성공: (284807, 31)


In [5]:
# 2. 데이터 전처리
# 2.1 Time 컬럼 삭제 , 데이터,타겟 분리
df, y_target = pp.split_features_target(raw_df, cols= 'Time')
df.shape, y_target.shape

((284807, 29), (284807,))

In [6]:
# 2.2 이상치를 경계값으로 치환
df = pp.cap_outliers(df)

In [7]:
clf = setup(
    data=pd.concat([df, y_target], axis=1),
    target=y_target.name,
    normalize=True,
    feature_selection=True,
    session_id=23,
    verbose=False
)

In [8]:
from sklearn.svm import LinearSVC, SVC
from sklearn.calibration import CalibratedClassifierCV
class CalibratedLinearSVC:
    """PyCaret에서 SVM을 확률 기반으로 사용할 수 있도록 래핑"""

    def __init__(self, **kwargs):
        self.base = LinearSVC(**kwargs)
        self.model = None

    def fit(self, X, y):
        self.model = CalibratedClassifierCV(self.base, cv=3)
        self.model.fit(X, y)
        return self

    def predict(self, X):
        return self.model.predict(X)

    def predict_proba(self, X):
        return self.model.predict_proba(X)

    def get_params(self, deep=True):
        return self.base.get_params()

    def set_params(self, **params):
        self.base.set_params(**params)
        return self

In [9]:
def to_grid(params, allowed_keys=None):
    """PyCaret 튜닝 가능한 grid 형태로 변환"""
    if allowed_keys:
        params = {k: v for k, v in params.items() if k in allowed_keys}
    return {k: [v] if not isinstance(v, (list, tuple)) else v for k, v in params.items()}

In [10]:
cat_params = copy.deepcopy(HP.cb_best_params)
cat_params['verbose'] = False
cat_grid = to_grid(cat_params)

# SVC 파라미터 → PyCaret용 안전 필터링
svc_allowed = SVC().get_params().keys()
safe_grid = to_grid(HP.svc_rbf_basic_params, allowed_keys=svc_allowed)

# LSVC Grid
lsvc_params = {"C": 1.0, "class_weight": "balanced", "max_iter": 3000}
lsvc_grid = to_grid(lsvc_params)

In [ ]:
from HP import mlp_basic_params 

# Notebook에서 덮어쓰기
mlp_basic_params["hidden_layer_sizes"] = [(64, 32)]
mlp_basic_params["activation"] = ["relu"]
mlp_basic_params["solver"] = ["adam"]
mlp_basic_params["alpha"] = [1e-4]
mlp_basic_params["batch_size"] = [256]
mlp_basic_params["learning_rate"] = ["adaptive"]
mlp_basic_params["max_iter"] = [50]
mlp_basic_params["random_state"] = [23]

In [11]:
# PyCaret 쓸 때 X_features를 따로 변수로 들고 있을 필요X
# PyCaret은 setup() 안에서 자동으로 Feature/Target 분리.
# LSVC랑 SVC(RBF)**를 둘 다 'svm'으로 처리했기 때문에 
# → PyCaret에서는 그 둘이 같은 모델 취급되면서 11개가 된 것.

In [12]:
tuned_models = {}

tuned_models['cat'] = tune_model(create_model('catboost'), custom_grid=cat_grid)
tuned_models['dt'] = tune_model(create_model('dt'), custom_grid=to_grid(HP.dt_basic_params))
tuned_models['gb'] = tune_model(create_model('gbc'), custom_grid=to_grid(HP.gb_best_params))
tuned_models['lgbm'] = tune_model(create_model('lightgbm'), custom_grid=to_grid(HP.lgbm_best_param2))
tuned_models['lr'] = tune_model(create_model('lr'), custom_grid=to_grid(HP.lr_best_params))
tuned_models['mlp'] = tune_model(create_model('mlp'), custom_grid=to_grid(HP.mlp_basic_params))
tuned_models['rf'] = tune_model(create_model('rf'), custom_grid=to_grid(HP.rf_best_params))
tuned_models['sgd'] = tune_model(create_model('sgd'), custom_grid=to_grid(HP.sgd_best_params))
tuned_models['xgb'] = tune_model(create_model('xgboost'), custom_grid=to_grid(HP.xgb_best_params))

lsvc_model = create_model(custom_model=CalibratedLinearSVC(), verbose=False)
tuned_models['lsvc'] = tune_model(lsvc_model, custom_grid=lsvc_grid)

print("\n🎯 튜닝 완료! 모델 성능 비교:\n")
results = pull()
results_sorted = results.sort_values(by="AUC", ascending=False)
results_sorted

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9992,0.9702,0.6286,0.8800,0.7333,0.7329,0.7434
1,0.9993,0.9828,0.7429,0.8667,0.8000,0.7997,0.8021
2,0.9990,0.8438,0.4286,1.0000,0.6000,0.5996,0.6543
3,0.9991,0.9667,0.6000,0.8750,0.7119,0.7115,0.7242
4,0.9996,0.9916,0.8824,0.9091,0.8955,0.8953,0.8954
5,0.9993,0.9927,0.7353,0.8621,0.7937,0.7933,0.7958
6,0.9989,0.9308,0.6765,0.6970,0.6866,0.6860,0.6861
7,0.9994,0.9731,0.7059,0.9231,0.8000,0.7997,0.8069
8,0.9988,0.9430,0.5588,0.7037,0.6230,0.6224,0.6265


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9988,0.9897,0.5143,0.7200,0.6000,0.5994,0.6079
1,0.9994,0.9831,0.7714,0.8710,0.8182,0.8179,0.8194
2,0.9989,0.8892,0.4571,0.8421,0.5926,0.5921,0.6200
3,0.9991,0.9684,0.5429,0.9048,0.6786,0.6781,0.7004
4,0.9995,0.9984,0.7647,0.9630,0.8525,0.8522,0.8579
5,0.9993,0.9971,0.7059,0.8571,0.7742,0.7738,0.7775
6,0.9990,0.9522,0.7059,0.7273,0.7164,0.7159,0.7160
7,0.9993,0.9879,0.6765,0.9200,0.7797,0.7793,0.7886
8,0.9989,0.9609,0.5294,0.7500,0.6207,0.6202,0.6296


Fitting 10 folds for each of 1 candidates, totalling 10 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9987,0.7997,0.6000,0.6364,0.6176,0.6170,0.6173
1,0.9989,0.8853,0.7429,0.6667,0.7027,0.7022,0.7032
2,0.9979,0.6852,0.3714,0.4062,0.3881,0.3870,0.3874
3,0.9987,0.8425,0.6857,0.6154,0.6486,0.6480,0.6489
4,0.9982,0.8523,0.7059,0.4898,0.5783,0.5775,0.5872
5,0.9992,0.8675,0.7353,0.8065,0.7692,0.7689,0.7697
6,0.9986,0.8672,0.7353,0.5814,0.6494,0.6487,0.6532
7,0.9989,0.8380,0.6765,0.6970,0.6866,0.6860,0.6861
8,0.9983,0.8083,0.6176,0.5122,0.5600,0.5592,0.5616


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9987,0.7997,0.6000,0.6364,0.6176,0.6170,0.6173
1,0.9989,0.8853,0.7429,0.6667,0.7027,0.7022,0.7032
2,0.9979,0.6852,0.3714,0.4062,0.3881,0.3870,0.3874
3,0.9987,0.8425,0.6857,0.6154,0.6486,0.6480,0.6489
4,0.9982,0.8523,0.7059,0.4898,0.5783,0.5775,0.5872
5,0.9992,0.8675,0.7353,0.8065,0.7692,0.7689,0.7697
6,0.9986,0.8672,0.7353,0.5814,0.6494,0.6487,0.6532
7,0.9989,0.8380,0.6765,0.6970,0.6866,0.6860,0.6861
8,0.9983,0.8083,0.6176,0.5122,0.5600,0.5592,0.5616


...

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.9987,0.9904,0.4571,0.7273,0.5614,0.5608,0.5760
1,0.9994,0.9793,0.7714,0.8710,0.8182,0.8179,0.8194
2,0.9989,0.8745,0.4286,0.8824,0.5769,0.5764,0.6145
3,0.9990,0.9597,0.6000,0.7778,0.6774,0.6769,0.6826
4,0.9989,0.9891,0.4706,0.8421,0.6038,0.6033,0.6291
5,0.9993,0.9913,0.7353,0.8621,0.7937,0.7933,0.7958
6,0.9989,0.9499,0.6765,0.6765,0.6765,0.6759,0.6759
7,0.9994,0.9863,0.7059,0.9231,0.8000,0.7997,0.8069
8,0.9986,0.9612,0.3235,0.6875,0.4400,0.4394,0.4710


,,
,,
Initiated,. . . . . . . . . . . . . . . . . .,19:04:13
Status,. . . . . . . . . . . . . . . . . .,Searching Hyperparameters
Estimator,. . . . . . . . . . . . . . . . . .,MLP Classifier


Processing:   0%|          | 0/7 [00:00<?, ?it/s]

Fitting 10 folds for each of 2 candidates, totalling 20 fits


ValueError: 
All the 20 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
20 fits failed with the following error:
Traceback (most recent call last):
  File "c:\ProgramData\anaconda3\envs\pycaret_env\lib\site-packages\sklearn\model_selection\_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\ProgramData\anaconda3\envs\pycaret_env\lib\site-packages\pycaret\internal\pipeline.py", line 279, in fit
    clone(self.steps[-1][1]), X, y, **last_step_params["fit"]
  File "c:\ProgramData\anaconda3\envs\pycaret_env\lib\site-packages\sklearn\base.py", line 90, in clone
    return estimator.__sklearn_clone__()
  File "c:\ProgramData\anaconda3\envs\pycaret_env\lib\site-packages\sklearn\base.py", line 296, in __sklearn_clone__
    return _clone_pa

In [15]:
# BestOpt 찾고 나서 스케일적용 버전 만들어서 모델링하기 
# 2.5 StandardScaler 적용
X_train_sscaled, X_test_sscaled, scaler = pp.scale_data(X_train, X_test)


In [16]:
# 2.6 robustScaler 적용

rscaler = RobustScaler()
X_train_rscaled = rscaler.fit_transform(X_train)   # 학습 데이터로 fit + transform
X_test_rscaled = rscaler.transform(X_test)         # 테스트 데이터는 transform만


In [ ]:
# 1️⃣ **Hard Voting - 다양성 극대화 조합** (claude 추천 1순위)
# ensemble_1 = {
#     'name': 'Diverse Hard Voting',
#     'method': 'VotingClassifier (hard)',
#     'models': [
#         'XGBoost',           # Tree-based, scale_pos_weight
#         'LightGBM',          # Tree-based, 빠른 학습
#         'LogisticRegression', # Linear, SMOTE
#         'SGD',               # Linear, 원본 데이터
#         'RandomForest'       # Bagging 기반
#     ],
#     'voting': 'hard',
#     'weights': None,
#     'reason': '다양한 알고리즘 타입(Boosting, Linear, Bagging) 혼합으로 편향 최소화'
# }
# 구현 예시
from sklearn.ensemble import VotingClassifier

ensemble_1_model = VotingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('lgbm', lgbm_model),
        ('lr', lr_model),
        ('sgd', sgd_model),
        ('rf', rf_model)
    ],
    voting='hard'
)

ensemble_1_model.fit(X_train, y_train)
y_pred = ensemble_1_model.predict(X_test)

In [ ]:
# 2️⃣ **Soft Voting - 확률 기반 앙상블**
# ensemble_2 = {
#     'name': 'Probability Weighted Soft Voting',
#     'method': 'VotingClassifier (soft)',
#     'models': [
#         'CatBoost',          # 높은 AUC 성능
#         'XGBoost',           # robust boosting
#         'LightGBM',          # 빠른 수렴
#         'LogisticRegression', # 확률 calibration 우수
#         'MLPClassifier'      # 비선형 패턴 학습
#     ],
#     'voting': 'soft',
#     'weights': [1.2, 1.2, 1.1, 1.0, 0.9],  # 성능 기반 가중치
#     'reason': '확률 기반 투표로 불확실성 처리, 고성능 모델에 가중치 부여'
# }
# 구현 예시
ensemble_2_model = VotingClassifier(
    estimators=[
        ('catboost', catboost_model),
        ('xgb', xgb_model),
        ('lgbm', lgbm_model),
        ('lr', lr_model),
        ('mlp', mlp_model)
    ],
    voting='soft',
    weights=[1.2, 1.2, 1.1, 1.0, 0.9]
)

ensemble_2_model.fit(X_train, y_train)
y_proba = ensemble_2_model.predict_proba(X_test)[:, 1]

In [ ]:
# 3️⃣ **Stacking - 2-Level Meta Learner**
# ensemble_3 = {
#     'name': 'Two-Level Stacking',
#     'method': 'StackingClassifier',
#     'base_models': [
#         'CatBoost',          # Level 1
#         'XGBoost',           # Level 1
#         'LightGBM',          # Level 1
#         'RandomForest',      # Level 1
#         'SVM_rbf',         # Level 1 - 비선형 경계
#         'SGD'                # Level 1 - 선형 경계
#     ],
#     'meta_model': 'LogisticRegression (kjh)',  # Level 2
#     'cv': 5,
#     'reason': 'Base 모델들의 예측을 meta-learner가 학습하여 최적 조합 발견'
# }
# 구현 예시
from sklearn.ensemble import StackingClassifier

ensemble_3_model = StackingClassifier(
    estimators=[
        ('catboost', catboost_model),
        ('xgb', xgb_model),
        ('lgbm', lgbm_model),
        ('rf', rf_model),
        ('svm', svm_rbf_model),
        ('sgd', sgd_model)
    ],
    final_estimator=LogisticRegression(class_weight='balanced'),
    cv=5,
    stack_method='predict_proba'
)

ensemble_3_model.fit(X_train, y_train)
y_pred = ensemble_3_model.predict(X_test)

In [ ]:
# 4️⃣ **Boosting 특화 앙상블**
# ensemble_4 = {
#     'name': 'Boosting Power Ensemble',
#     'method': 'Weighted Average (Custom)',
#     'models': [
#         'CatBoost',          # 범주형 변수 강점
#         'XGBoost',           # 고속 학습
#         'LightGBM',          # 메모리 효율
#         'GradientBoosting'   # sklearn 안정성
#     ],
#     'weights': [0.30, 0.30, 0.25, 0.15],  # AUC 기준 가중치
#     'threshold': 0.3,  # 사기 탐지 임계값 조정
#     'reason': 'Boosting 계열만으로 구성, 순차 학습의 강점 극대화'
# }
# 구현 예시

# 각 모델의 확률 예측
catboost_proba = catboost_model.predict_proba(X_test)[:, 1]
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
lgbm_proba = lgbm_model.predict_proba(X_test)[:, 1]
gb_proba = gb_model.predict_proba(X_test)[:, 1]

# 가중 평균
weights = [0.30, 0.30, 0.25, 0.15]
ensemble_4_proba = (
    weights[0] * catboost_proba +
    weights[1] * xgb_proba +
    weights[2] * lgbm_proba +
    weights[3] * gb_proba
)

# 임계값 조정 (사기 탐지 최적화)
threshold = 0.3
y_pred = (ensemble_4_proba >= threshold).astype(int)

In [ ]:
# 5️⃣ **Hybrid - Tree + Linear 균형 조합**
# ensemble_5 = {
#     'name': 'Tree-Linear Hybrid Voting',
#     'method': 'VotingClassifier (soft)',
#     'models': [
#         # Tree 기반 (60%)
#         'XGBoost',           # 20%
#         'LightGBM',          # 20%
#         'RandomForest',      # 20%
        
#         # Linear 기반 (40%)
#         'LogisticRegression', # 20%
#         'SGD',               # 10%
#         'lsvc'       # 10%
#     ],
#     'voting': 'soft',
#     'weights': [1.2, 1.2, 1.2, 1.0, 0.8, 0.8],
#     'reason': 'Tree의 비선형 포착 + Linear의 일반화 능력 결합'
# }

### ChatGpt 추천 조합

| 조합 번호 | 구성                         | 특징              |
| ----- | -------------------------- | --------------- |
| **1** | CatBoost + XGB + LGBM + LR | 성능 최상위 전천후      |
| **2** | LGBM + RF + MLP            | 구조적 다양성 최고      |
| **3** | CatBoost + GB + SVM(RBF)   | Recall 최적화      |
| **4** | XGB + LR + SVM(linear)     | 단순·안정·일관된 결정 경계 |
| **5** | RF + SGD + MLP + LR        | 전통 ML 메타 스택     |

In [ ]:
# ===========================================================
# 2. Base Models (HyperOpt 결과 삽입)
# ===========================================================

# HP 이용

# Meta Model
# meta_model = LogisticRegression(max_iter=2000)


# ===========================================================
# 3. Stacking 함수 직접 작업할때 예시
# ===========================================================
# def stacking_predict(X_train, y_train, X_test, n_folds=5):

#     kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

#     # Level-1 예측값 저장
#     oof_cat, oof_xgb, oof_lgb = (
#         np.zeros(len(X_train)),
#         np.zeros(len(X_train)),
#         np.zeros(len(X_train)),
#     )
#     test_cat, test_xgb, test_lgb = (
#         np.zeros(len(X_test)),
#         np.zeros(len(X_test)),
#         np.zeros(len(X_test)),
#     )

#     for fold, (trn_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
#         print(f"Fold {fold+1}/{n_folds}")

#         X_tr, X_val = X_train.iloc[trn_idx], X_train.iloc[val_idx]
#         y_tr, y_val = y_train.iloc[trn_idx], y_train.iloc[val_idx]

#         # ---------------------------------------------------
#         # CatBoost
#         # ---------------------------------------------------
#         model_cat = CatBoostClassifier(**cat_params)
#         model_cat.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False)

#         oof_cat[val_idx] = model_cat.predict_proba(X_val)[:, 1]
#         test_cat += model_cat.predict_proba(X_test)[:, 1] / n_folds

#         # ---------------------------------------------------
#         # XGBoost
#         # ---------------------------------------------------
#         model_xgb = XGBClassifier(**xgb_params)
#         model_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

#         oof_xgb[val_idx] = model_xgb.predict_proba(X_val)[:, 1]
#         test_xgb += model_xgb.predict_proba(X_test)[:, 1] / n_folds

#         # ---------------------------------------------------
#         # LightGBM
#         # ---------------------------------------------------
#         model_lgb = LGBMClassifier(**lgbm_params)
#         model_lgb.fit(X_tr, y_tr,
#                       eval_set=[(X_val, y_val)],
#                       verbose=False)

#         oof_lgb[val_idx] = model_lgb.predict_proba(X_val)[:, 1]
#         test_lgb += model_lgb.predict_proba(X_test)[:, 1] / n_folds

#     # =======================================================
#     # Meta Model 학습
#     # =======================================================
#     train_meta = np.vstack([oof_cat, oof_xgb, oof_lgb]).T
#     test_meta = np.vstack([test_cat, test_xgb, test_lgb]).T

#     scaler = StandardScaler()
#     train_meta_scaled = scaler.fit_transform(train_meta)
#     test_meta_scaled = scaler.transform(test_meta)

#     meta_model.fit(train_meta_scaled, y_train)
#     meta_oof = meta_model.predict_proba(train_meta_scaled)[:, 1]

#     print("Stacking Level-2 AUC:", roc_auc_score(y_train, meta_oof))

#     return test_meta_scaled, meta_model


# ===========================================================
# 4. 실행 예시
# ===========================================================
# X_train, y_train, X_test는 이미 로딩된 상태라고 가정
# final_test_meta, final_meta_model = stacking_predict(X_train, y_train, X_test)


In [ ]:
def create_lr(best_params):
    """
    LogisticRegression 모델을 HyperOpt/Optuna로 찾은 best_params 기반으로 생성하는 함수.

    이 함수는 LogisticRegression의 solver와 penalty 조합이 유효한지 검사하고,
    유효하지 않은 조합이 들어올 경우 자동으로 수정하여 안전하게 모델을 생성한다.

    Parameters
    ----------
    best_params : dict
        HyperOpt 또는 Optuna로 최적화한 LogisticRegression의 최적 파라미터 딕셔너리.
        예: {"C": 0.1, "solver": "liblinear", "penalty": "l1"}

    Returns
    -------
    LogisticRegression
        최종적으로 검증된 파라미터로 생성된 LogisticRegression 모델.

    Notes
    -----
    - solver에 따라 사용할 수 있는 penalty 종류가 다르기 때문에,
      최적화 결과가 잘못된 조합을 반환할 가능성이 있음.
    - 안전성을 위해 직접 검증한 후 잘못된 penalty는 'l2'로 자동 변경한다.
    - 변경이 발생하면 경고 메시지를 출력한다.
    """

    # 최적화된 solver, penalty 값을 가져오고 기본값 설정
    solver = best_params.get('solver', 'liblinear')
    penalty = best_params.get('penalty', 'l2')

    # solver별로 허용되는 penalty 목록 정의
    valid_penalties = {
        'liblinear': ['l1', 'l2'],
        'lbfgs': ['l2', 'none'],
        'saga': ['l1', 'l2', 'elasticnet', 'none'],
        'newton-cg': ['l2', 'none'],
    }

    # penalty가 solver에 맞지 않으면 자동 수정
    if penalty not in valid_penalties.get(solver, []):
        print(f"[WARN] penalty '{penalty}' is incompatible with solver '{solver}'. Using 'l2'")
        best_params['penalty'] = 'l2'

    # 유효한 파라미터로 LogisticRegression 모델 생성
    return LogisticRegression(**best_params)
# eof -----------------------------------------------------------


In [ ]:
# ======================================
# 🔷 Stacking #1: CatBoost + XGB + LGBM
# ======================================

models = get_models()

estimators_1 = [
    ('cat', models['cat']),
    ('xgb', models['xgb']),
    ('lgbm', models['lgbm']),
]


stack_1 = StackingClassifier(
    estimators=estimators_1,
    final_estimator=create_lr(**HP.lr_best_params),
    stack_method='predict_proba',
    cv=5,
    n_jobs=-1
)
option_name = 'cat+xgb+lgbm_ho_best'
results[option_name] = uu.get_model_train_eval(stack_1, f'{option_name}', X_train, X_test, y_train, y_test)


In [ ]:
# ======================================
# 🔷 Stacking #2: LGBM + RF + MLP
# ======================================

models = get_models()

estimators_2 = [
    ('lgbm', models['lgbm']),
    ('rf', models['rf']),
    ('mlp', models['mlp']),
]

stack_2 = StackingClassifier(
    estimators=estimators_2,
    final_estimator=create_lr(**HP.lr_best_params),
    stack_method='predict_proba',
    cv=5,
    n_jobs=-1
)
option_name = 'lgbm+rf+mlp_ho_best'
results[option_name] = uu.get_model_train_eval(stack_2, f'{option_name}', X_train, X_test, y_train, y_test)


In [ ]:
# ======================================
# 🔷 Stacking #3: CatBoost + GB + SVM(RBF)
# ======================================

estimators_3 = [
    ('cat', models['cat']),
    ('gb', models['gb']),
    ('svm_rbf', models['svm_rbf']),
]

stack_3 = StackingClassifier(
    estimators=estimators_3,
    final_estimator=create_lr(**HP.lr_best_params),
    stack_method='predict_proba',
    cv=5,
    n_jobs=-1
)

In [ ]:
# ======================================
# 🔷 Stacking #4: XGB + LR + SVM(linear)
# ======================================

# LR은 메타모델로 사용하는 게 더 좋아서 base에는 넣지 않음

est_lr = LogisticRegression(
    penalty='l2',
    C=1.0,
    solver='liblinear',
    class_weight='balanced'
)

estimators_4 = [
    ('xgb', models['xgb']),
    ('lsvc', models['lsvc']),
]

stack_4 = StackingClassifier(
    estimators=estimators_4,
    final_estimator=est_lr,
    stack_method='predict_proba',
    cv=5,
    n_jobs=-1
)

In [ ]:
# ======================================
# 🔷 Stacking #5: RF + SGD + MLP + LR
# ======================================

estimators_5 = [
    ('rf', models['rf']),
    ('sgd', models['sgd']),
    ('mlp', models['mlp']),
]

stack_5 = StackingClassifier(
    estimators=estimators_5,
    # final_estimator=LogisticRegression(class_weight='balanced', solver='liblinear'),
    final_estimator=create_lr(**HP.lr_best_params),
    stack_method='predict_proba',
    cv=5,
    n_jobs=-1
)

In [ ]:
# ===========================================
# 🔷 5개 Stack 모델을 Soft Voting으로 결합
# ===========================================

from sklearn.ensemble import VotingClassifier

# VotingClassifier는 원래 predict_proba를 지원하는 모델만 가능
# 우리 스택 모델들은 모두 predict_proba 지원하므로 문제 없음
weights_desc = '''
| 스택                         | 추천 이유     | weight |
| -------------------------- | --------- | ------ |
| Stack #1 (Boosting+LR)     | 대부분 최고 성능 | **3**  |
| Stack #3 (CatBoost+GB+SVM) | Recall 강함 | **2**  |
| Stack #2 (LGBM+RF+MLP)     | 안정적       | **2**  |
| Stack #4 (XGB+LR+SVM)      | 선형 경계 보정  | **1**  |
| Stack #5 (RF+SGD+MLP)      | 편향 다양성 확보 | **1**  |
'''

voting_ensemble = VotingClassifier(
    estimators=[
        ('stack1', stack_1),
        ('stack2', stack_2),
        ('stack3', stack_3),
        ('stack4', stack_4),
        ('stack5', stack_5),
    ],
    voting='soft',          # 🔥 중요: 확률 기반 soft voting
    weights=[3, 2, 2, 1, 1], # 가중치 
    n_jobs=-1
)

In [ ]:
# 시각화
mo.model_metrics_graph(results, 'LGBM 데이터별 성능지표 비교')